In [17]:
import numpy as np
import random
import math

In [19]:
# --- Safe Operations ---

EPS = 1e-6  # small constant to avoid divide-by-zero and overflow

def safe_div(a, b):
    """Safe division: returns a / b if |b| > EPS, otherwise returns 0."""
    try:
        return a / (b if abs(b) > EPS else EPS)
    except Exception:
        return 0.

def safe_log(a):
    """Safe natural log: returns log(|a|) if a is not too close to zero, otherwise returns 0."""
    if abs(a) < EPS:
        return 0.
    return np.log(abs(a))

# --- Primitive Set ---

# Define a set of functions that can operate elementwise on numpy arrays
primitive_set = {
    'add': {'func': np.add, 'arity': 2, 'repr': '+'},
    'sub': {'func': np.subtract, 'arity': 2, 'repr': '-'},
    'mul': {'func': np.multiply, 'arity': 2, 'repr': '*'},
    'div': {'func': safe_div, 'arity': 2, 'repr': '/'},
    'sin': {'func': np.sin, 'arity': 1, 'repr': 'sin'},
    'cos': {'func': np.cos, 'arity': 1, 'repr': 'cos'},
    'tan': {'func': np.tan, 'arity': 1, 'repr': 'tan'},
    'exp': {'func': np.exp, 'arity': 1, 'repr': 'exp'},
    'log': {'func': safe_log, 'arity': 1, 'repr': 'log'},
    # Add other functions as needed...
}

# We also treat the input variables as terminals. In our case, since the shape of x is (n_vars, n_samples), we refer to them by index, e.g. x[0].
terminal_set = ['x[%d]' % i for i in range(10)]  # allocate up to 10 possible variables, will use as many as needed
# Optionally, add constants as terminals:
constant_pool = [str(round(random.uniform(-5, 5), 2)) for _ in range(5)]

In [21]:
# --- Expression Tree Structure ---

class Node:
    def __init__(self, content, children=None):
        self.content = content  # either a key for primitive_set or a terminal (variable or constant)
        self.children = children if children is not None else []

    def is_terminal(self):
        return len(self.children) == 0

    def evaluate(self, x):
        """Recursively evaluate the expression tree.
           x is expected to be a numpy array where each row is an input variable."""
        # If node is a terminal, interpret it.
        if self.is_terminal():
            # If the content is of form 'x[i]', extract the row.
            if isinstance(self.content, str) and self.content.startswith('x['):
                # Evaluate expression, e.g., "x[0]" returns the first row
                index = int(self.content[2:-1])
                return x[index]
            else:
                # It is a constant string, convert to float and return a numpy array of constant
                return float(self.content) * np.ones(x.shape[1])
        else:
            # Get the function from the primitive set.
            op = primitive_set[self.content]
            # Recursively evaluate children:
            args = [child.evaluate(x) for child in self.children]
            # Apply the operator in a vectorized way.
            return op['func'](*args)

    def __str__(self):
        """Return a string representation of the expression."""
        if self.is_terminal():
            return str(self.content)
        else:
            op_repr = primitive_set[self.content]['repr']
            if primitive_set[self.content]['arity'] == 1:
                return f"{op_repr}({self.children[0]})"
            elif primitive_set[self.content]['arity'] == 2:
                return f"({self.children[0]} {op_repr} {self.children[1]})"
            else:
                return f"{self.content}(" + ", ".join(str(child) for child in self.children) + ")"

In [ ]:
# --- Generate Random Expression Trees ---

def generate_random_tree(max_depth, current_depth=0, n_vars=2):
    """Generate a random expression tree with a maximum depth."""
    if current_depth >= max_depth or (current_depth > 0 and random.random() < 0.2):
        # Terminal: either a variable (if within dimension) or a constant
        if random.random() < 0.7:
            # Choose one of the available input variables
            var_index = random.randint(0, n_vars - 1)
            return Node(f'x[{var_index}]')
        else:
            # Use a random constant from the pool
            return Node(random.choice(constant_pool))
    else:
        # Choose a random primitive operation from the set
        op_key = random.choice(list(primitive_set.keys()))
        arity = primitive_set[op_key]['arity']
        children = [generate_random_tree(max_depth, current_depth + 1, n_vars) for _ in range(arity)]
        return Node(op_key, children)

In [ ]:
# --- Genetic Operators: Crossover and Mutation ---

def mutate(node, max_depth, n_vars):
    """Mutate a tree node by randomly replacing a subtree."""
    if random.random() < 0.1:  # mutation probability at this node level
        return generate_random_tree(max_depth, n_vars=n_vars)
    if not node.is_terminal():
        new_children = [mutate(child, max_depth, n_vars) for child in node.children]
        return Node(node.content, new_children)
    return node

def crossover(node1, node2):
    """Swap a random subtree between node1 and node2."""
    if random.random() < 0.1:
        return node2
    if not node1.is_terminal() and not node2.is_terminal():
        new_children = []
        for child in node1.children:
            # With a certain probability, replace this child with a randomly chosen subtree from node2.
            if random.random() < 0.5:
                new_children.append(random.choice(flatten_tree(node2)))
            else:
                new_children.append(child)
        return Node(node1.content, new_children)
    return node1

def flatten_tree(node):
    """Return a list of all nodes in the tree."""
    nodes = [node]
    if not node.is_terminal():
        for child in node.children:
            nodes.extend(flatten_tree(child))
    return nodes

In [ ]:
# --- Fitness Function ---

def fitness(individual: Node, x: np.ndarray, y: np.ndarray):
    """Compute the Mean Squared Error (MSE) of an individual's expression on the data."""
    try:
        y_pred = individual.evaluate(x)
        # Ensure y_pred has the same shape as y.
        if y_pred.shape != y.shape:
            return np.inf
        mse = np.mean((y - y_pred)**2)
        if np.isnan(mse):
            return np.inf
        return mse
    except Exception:
        return np.inf